# NFPM Feature Overview

Notebook d'exploration des positions NFT mappées à la pool suivie. On s'appuie sur les marts NFPM pour regarder l'état courant, le lifecycle, les fees et le PnL.

## Prérequis

Les visualisations ci-dessous supposent que ces modèles sont matérialisés dans ClickHouse local:

- `mart_nfpm_positions_current`
- `mart_nfpm_position_lifecycle`
- `mart_nfpm_position_pnl_current`
- `mart_tvl_current` pour le contexte pool

Si l'inventaire ci-dessous remonte des tables NFPM à `0 rows`, rerun local utile:

```bash
cd /home/gharbi/defind-analytics/dbt/defind_analytics
../../.venv/bin/dbt build --target local --select int_nfpm_pool_mints int_nfpm_token_mint_actions int_nfpm_positions int_nfpm_position_liquidity_events int_nfpm_position_collect_events int_nfpm_position_collect_decomposition int_nfpm_position_transfer_rollup int_nfpm_position_collect_rollup mart_nfpm_positions_current mart_nfpm_position_lifecycle mart_nfpm_position_pnl_current
```

Tu peux sélectionner la pool via `POOL_ADDRESS` dans l'environnement si besoin.

In [1]:
import os
from decimal import Decimal
from textwrap import dedent

import clickhouse_connect

try:
    from IPython.display import display
except ModuleNotFoundError:
    def display(obj):
        print(obj)

try:
    import pandas as pd
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        'pandas is required. Run: %pip install pandas clickhouse-connect plotly jupyterlab'
    ) from exc

try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ModuleNotFoundError:
    PLOTLY_AVAILABLE = False
    print('Plotly not available. Run: %pip install plotly')

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 200)


In [2]:
CLICKHOUSE_HOST = os.getenv('CLICKHOUSE_HOST', 'localhost')
CLICKHOUSE_PORT = int(os.getenv('CLICKHOUSE_PORT', '8124'))
CLICKHOUSE_USER = os.getenv('CLICKHOUSE_USER', 'default')
CLICKHOUSE_PASSWORD = os.getenv('CLICKHOUSE_PASSWORD', '1234')
CLICKHOUSE_DATABASE = os.getenv('CLICKHOUSE_DATABASE', 'analytics')

client = clickhouse_connect.get_client(
    host=CLICKHOUSE_HOST,
    port=CLICKHOUSE_PORT,
    username=CLICKHOUSE_USER,
    password=CLICKHOUSE_PASSWORD,
    database=CLICKHOUSE_DATABASE,
)

print(f'Connected to ClickHouse database={CLICKHOUSE_DATABASE} on {CLICKHOUSE_HOST}:{CLICKHOUSE_PORT}')


Connected to ClickHouse database=analytics on localhost:8124


In [3]:
TIME_COLUMNS = [
    'date',
    'hour',
    'mint_block_time',
    'ownership_block_time',
    'last_transfer_block_time',
    'last_liquidity_event_block_time',
    'last_collect_block_time',
    'last_position_activity_time',
    'first_secondary_transfer_block_time',
    'last_secondary_transfer_block_time',
    'burn_transfer_block_time',
    'first_liquidity_event_block_time',
    'first_increase_block_time',
    'last_increase_block_time',
    'first_decrease_block_time',
    'last_decrease_block_time',
    'first_collect_block_time',
    'entry_snapshot_block_time',
]

def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    for column in TIME_COLUMNS:
        if column in df.columns:
            df[column] = pd.to_datetime(df[column])
    for column in df.columns:
        if df[column].dtype == 'object':
            series = df[column].dropna()
            if not series.empty and series.map(lambda value: isinstance(value, Decimal)).all():
                df[column] = df[column].astype(float)
    return df

def query_df(sql: str, parameters: dict | None = None) -> pd.DataFrame:
    df = client.query_df(sql, parameters=parameters or {})
    df = normalize_df(df)
    print(f'{len(df):,} rows')
    return df

def plot_guard() -> bool:
    if not PLOTLY_AVAILABLE:
        print('Plotly unavailable. Install with: %pip install plotly')
        return False
    return True


## 1. Pool et couverture NFPM

In [4]:
available_pools = query_df(dedent(
    '''
    SELECT DISTINCT pool
    FROM (
        SELECT pool FROM mart_nfpm_positions_current
        UNION ALL
        SELECT pool FROM mart_tvl_current
    )
    ORDER BY pool
    '''
))
available_pools


1 rows


,pool
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640


In [5]:
POOL = os.getenv('POOL_ADDRESS', available_pools.iloc[0]['pool'] if not available_pools.empty else None)
assert POOL, 'No pool found in mart_tvl_current or mart_nfpm_positions_current.'
print(f'Using pool: {POOL}')


Using pool: 0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640


In [6]:
nfpm_inventory = query_df(dedent(
    '''
    SELECT
        model,
        rows,
        min_time,
        max_time
    FROM (
        SELECT 'mart_nfpm_positions_current' AS model, count() AS rows, min(mint_block_time) AS min_time, max(last_position_activity_time) AS max_time FROM mart_nfpm_positions_current WHERE pool = %(pool)s
        UNION ALL
        SELECT 'mart_nfpm_open_positions_current' AS model, count() AS rows, min(mint_block_time) AS min_time, max(last_position_activity_time) AS max_time FROM mart_nfpm_open_positions_current WHERE pool = %(pool)s
        UNION ALL
        SELECT 'mart_nfpm_position_lifecycle' AS model, count() AS rows, min(mint_block_time) AS min_time, max(last_position_activity_time) AS max_time FROM mart_nfpm_position_lifecycle WHERE pool = %(pool)s
        UNION ALL
        SELECT 'mart_nfpm_position_pnl_current' AS model, count() AS rows, min(mint_block_time) AS min_time, max(last_position_activity_time) AS max_time FROM mart_nfpm_position_pnl_current WHERE pool = %(pool)s
        UNION ALL
        SELECT 'mart_tvl_current' AS model, count() AS rows, NULL AS min_time, NULL AS max_time FROM mart_tvl_current WHERE pool = %(pool)s
    )
    ORDER BY model
    '''
), {'pool': POOL})
nfpm_inventory


5 rows


,model,rows,min_time,max_time
0,mart_nfpm_open_positions_current,4058,2021-05-05 21:42:11,2026-04-06 11:24:59
1,mart_nfpm_position_lifecycle,114358,2021-05-05 21:42:11,2026-04-06 11:24:59
2,mart_nfpm_position_pnl_current,114358,2021-05-05 21:42:11,2026-04-06 11:24:59
3,mart_nfpm_positions_current,114358,2021-05-05 21:42:11,2026-04-06 11:24:59
4,mart_tvl_current,1,NaT,NaT


In [7]:
pool_context = query_df(dedent(
    '''
    SELECT
        pool,
        current_tick,
        price_token1_in_token0,
        total_tvl_token0,
        active_tvl_token0,
        out_of_range_tvl_token0,
        current_bin_tvl_token0
    FROM mart_tvl_current
    WHERE pool = %(pool)s
    '''
), {'pool': POOL})

nfpm_snapshot = query_df(dedent(
    '''
    SELECT
        count() AS positions_total,
        sum(is_open) AS open_positions,
        sum(coalesce(is_active, 0)) AS active_positions,
        sum(is_nft_burned) AS burned_positions,
        uniqExact(current_owner) AS current_owner_count,
        sum(current_position_token0) AS deployed_principal_token0,
        avgIf(range_width_ticks, is_open = 1) AS avg_open_width_ticks,
        medianIf(range_width_ticks, is_open = 1) AS median_open_width_ticks
    FROM mart_nfpm_positions_current
    WHERE pool = %(pool)s
    '''
), {'pool': POOL})

nfpm_pnl_snapshot = query_df(dedent(
    '''
    SELECT
        count() AS pnl_rows,
        sum(current_total_wealth_token0) AS total_current_wealth_token0,
        sum(current_principal_wealth_token0) AS total_principal_wealth_token0,
        sum(current_fee_wealth_token0) AS total_fee_wealth_token0,
        sum(current_collected_fee_wealth_token0) AS total_collected_fee_wealth_token0,
        sum(current_uncollected_fee_wealth_token0) AS total_uncollected_fee_wealth_token0,
        avgIf(observed_time_in_range_pct, observed_window_seconds > 0) AS avg_observed_time_in_range_pct,
        medianIf(pnl_vs_hodl_pct, pnl_vs_hodl_pct IS NOT NULL) AS median_pnl_vs_hodl_pct
    FROM mart_nfpm_position_pnl_current
    WHERE pool = %(pool)s
    '''
), {'pool': POOL})

display(pool_context)
display(nfpm_snapshot)
display(nfpm_pnl_snapshot)


1 rows
1 rows
1 rows


,pool,current_tick,price_token1_in_token0,total_tvl_token0,active_tvl_token0,out_of_range_tvl_token0,current_bin_tvl_token0
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,199587,2150.092004,9.362351e+07,7.460444e+07,1.901907e+07,336558.14942


,positions_total,open_positions,active_positions,burned_positions,current_owner_count,deployed_principal_token0,avg_open_width_ticks,median_open_width_ticks
0,114358,4058,2205,27831,22609,9.345948e+07,215994.620503,2730.0


,pnl_rows,total_current_wealth_token0,total_principal_wealth_token0,total_fee_wealth_token0,total_collected_fee_wealth_token0,total_uncollected_fee_wealth_token0,avg_observed_time_in_range_pct,median_pnl_vs_hodl_pct
0,114358,7.193793e+11,7.193793e+11,0.0,2.738792e+08,-2.738792e+08,NaN,-9.999785e-12


## 2. Positions courantes

In [8]:
positions_current = query_df(dedent(
    '''
    SELECT
        token_id,
        current_owner,
        tick_lower,
        tick_upper,
        range_width_ticks,
        toUInt8(is_open) AS is_open,
        toUInt8(coalesce(is_active, 0)) AS is_active,
        current_liquidity_share_of_initial,
        current_position_amount0,
        current_position_amount1,
        current_position_token0,
        toFloat64(total_principal_collected_amount0) AS total_principal_collected_amount0,
        toFloat64(total_fees_collected_amount0) AS total_fees_collected_amount0,
        toFloat64(price_token1_in_token0) AS current_price_token1_in_token0
    FROM mart_nfpm_positions_current
    WHERE pool = %(pool)s
    ORDER BY current_position_token0 DESC
    LIMIT 5000
    '''
), {'pool': POOL})

if not positions_current.empty:
    positions_current['position_status'] = positions_current.apply(
        lambda row: 'active' if row['is_active'] else ('open_out_of_range' if row['is_open'] else 'closed'),
        axis=1,
    )

positions_current.head(20)


5,000 rows


,token_id,current_owner,tick_lower,tick_upper,range_width_ticks,is_open,is_active,current_liquidity_share_of_initial,current_position_amount0,current_position_amount1,current_position_token0,total_principal_collected_amount0,total_fees_collected_amount0,current_price_token1_in_token0,position_status
0,517997,0x3c21f1a9daed29d769e81e3e6148b09a336e8b87,199380,200860,1480,1,1,4.050258,2.169352e+07,1689.657869,2.532644e+07,0.000000e+00,1.630849e+06,2150.092004,active
1,1199462,0x30c35950f2a8b8b76338bcd0b4d6fe67c98d2439,195610,209470,13860,1,1,1590.527811,6.335152e+06,1362.898631,9.265510e+06,0.000000e+00,4.848335e+04,2150.092004,active
2,514757,0x3c21f1a9daed29d769e81e3e6148b09a336e8b87,200150,201940,1790,1,0,1.055894,9.034560e+06,0.000000,9.034560e+06,0.000000e+00,6.146737e+05,2150.092004,open_out_of_range
3,305371,0x4b296808f414ab3775889fa2863e1d73f958a58e,194710,212350,17640,1,1,4.869826,3.747333e+06,799.592201,5.466530e+06,2.602583e+07,5.146853e+05,2150.092004,active
4,517984,0x3755ce6de02fdfc70798ed522f01641c3c70c220,199310,200870,1560,1,1,1.174268,3.896068e+06,402.043602,4.760499e+06,0.000000e+00,3.278112e+05,2150.092004,active
5,1204016,0x6c3c0d33de260dcd8cd6589475903de129705835,198320,202640,4320,1,1,1.000000,2.123733e+06,428.528893,3.045110e+06,0.000000e+00,5.063118e+04,2150.092004,active
6,1244018,0x8ba40d428f4570bd491d243fb8c8211e8d554e76,198910,200310,1400,1,1,1.000000,1.393650e+06,608.543227,2.702074e+06,0.000000e+00,1.894891e+04,2150.092004,active
7,514747,0x3755ce6de02fdfc70798ed522f01641c3c70c220,200150,201940,1790,1,0,1.030887,1.750105e+06,0.000000,1.750105e+06,0.000000e+00,1.216045e+05,2150.092004,open_out_of_range
8,1194729,0x75e126db7f730fada6c3b879413bea339a357e50,196950,206450,9500,1,1,1.000000,8.887041e+05,175.821962,1.266737e+06,0.000000e+00,0.000000e+00,2150.092004,active
9,1218440,0x37c86bd06d266a009d2de7c1263ec4ab7adde628,198490,200310,1820,1,1,1.045150,3.990090e+05,279.307051,9.995449e+05,0.000000e+00,2.243115e+04,2150.092004,active


In [9]:
if plot_guard() and not positions_current.empty:
    fig = px.histogram(
        positions_current,
        x='range_width_ticks',
        nbins=60,
        color='position_status',
        title=f'NFPM position width distribution - {POOL}',
        labels={'range_width_ticks': 'Width (ticks)', 'position_status': 'Status'},
    )
    fig.show()


In [10]:
if plot_guard() and not positions_current.empty:
    fig = px.scatter(
        positions_current,
        x='range_width_ticks',
        y='current_position_token0',
        color='position_status',
        hover_data=['token_id', 'current_owner', 'tick_lower', 'tick_upper'],
        title=f'NFPM deployed principal vs width - {POOL}',
        labels={'current_position_token0': 'Current deployed principal (token0)', 'range_width_ticks': 'Width (ticks)'},
        log_y=True,
    )
    fig.show()


In [11]:
owner_concentration = query_df(dedent(
    '''
    SELECT
        current_owner,
        count() AS position_count,
        sum(current_position_token0) AS deployed_principal_token0,
        sum(is_open) AS open_positions,
        sum(coalesce(is_active, 0)) AS active_positions
    FROM mart_nfpm_positions_current
    WHERE pool = %(pool)s
    GROUP BY current_owner
    ORDER BY deployed_principal_token0 DESC
    LIMIT 25
    '''
), {'pool': POOL})

owner_concentration.head(10)


25 rows


,current_owner,position_count,deployed_principal_token0,open_positions,active_positions
0,0x3c21f1a9daed29d769e81e3e6148b09a336e8b87,4,3.436100e+07,2,1
1,0x30c35950f2a8b8b76338bcd0b4d6fe67c98d2439,2,9.265510e+06,1,1
2,0x3755ce6de02fdfc70798ed522f01641c3c70c220,3,6.510604e+06,2,1
3,0x4b296808f414ab3775889fa2863e1d73f958a58e,1,5.466530e+06,1,1
4,0x6c3c0d33de260dcd8cd6589475903de129705835,1,3.045110e+06,1,1
5,0x8ba40d428f4570bd491d243fb8c8211e8d554e76,7,2.837600e+06,3,3
6,0x75e126db7f730fada6c3b879413bea339a357e50,3,1.266737e+06,1,1
7,0x37c86bd06d266a009d2de7c1263ec4ab7adde628,6,9.995449e+05,1,1
8,0xdd9bd62d1c08210fdda6f841ed5196b45a939625,54,8.600366e+05,1,1
9,0x9284205bb73efd293e310b12a37788114a7724a9,1,8.203807e+05,1,1


In [12]:
if plot_guard() and not owner_concentration.empty:
    owner_concentration['owner_short'] = owner_concentration['current_owner'].str.slice(0, 10)
    fig = px.bar(
        owner_concentration.sort_values('deployed_principal_token0'),
        x='deployed_principal_token0',
        y='owner_short',
        orientation='h',
        title=f'Top NFPM owners by deployed principal - {POOL}',
        labels={'deployed_principal_token0': 'Deployed principal (token0)', 'owner_short': 'Owner'},
    )
    fig.show()


## 3. Lifecycle

In [13]:
lifecycle = query_df(dedent(
    '''
    SELECT
        token_id,
        current_owner,
        lifecycle_status,
        transfer_count,
        collect_count,
        increase_count,
        decrease_count,
        observed_lifetime_seconds / 86400.0 AS observed_lifetime_days,
        time_to_first_collect_seconds / 86400.0 AS time_to_first_collect_days,
        time_to_close_seconds / 86400.0 AS time_to_close_days,
        current_position_token0
    FROM mart_nfpm_position_lifecycle
    WHERE pool = %(pool)s
    ORDER BY observed_lifetime_seconds DESC
    LIMIT 10000
    '''
), {'pool': POOL})
lifecycle.head(20)


10,000 rows


,token_id,current_owner,lifecycle_status,transfer_count,collect_count,increase_count,decrease_count,observed_lifetime_days,time_to_first_collect_days,time_to_close_days,current_position_token0
0,66907,0x7956e3ec749c83bf329dcdfd28f93c8caf949794,closed_unburned,1,3,1,3,1713.265012,0.023773,0.463194,0.00000
1,66955,0x7956e3ec749c83bf329dcdfd28f93c8caf949794,closed_unburned,1,1,1,1,1713.232558,3.416273,3.416273,0.00000
2,81150,0x8ca1b5e4cfec85f4a49bebd34a25550f758d6b50,open,2,1,3,0,1671.296725,1667.004502,NaN,1430.13199
3,110296,0x7956e3ec749c83bf329dcdfd28f93c8caf949794,closed_unburned,1,4,1,4,1669.847188,0.752500,19.192454,0.00000
4,122551,0x7956e3ec749c83bf329dcdfd28f93c8caf949794,closed_unburned,1,1,1,1,1646.523681,0.095347,0.095347,0.00000
5,9982,0x550dae2d4a0fedfd28babb9788af69fbd5b0b702,closed_unburned,0,1,1,1,1614.092477,1614.092477,1614.092477,0.00000
6,55536,0xabd1645db5ec528ab5d2a2ce9e851ad32e02e9a6,closed_unburned,0,1,1,1,1610.858634,1610.858634,1610.858634,0.00000
7,125086,0x000000000000000000000000000000000000dead,owner_burn_like,1,2,2,2,1571.943715,1.686389,2.262870,0.00000
8,125887,0x000000000000000000000000000000000000dead,owner_burn_like,1,3,3,3,1570.253032,0.147755,6.957083,0.00000
9,126074,0x000000000000000000000000000000000000dead,owner_burn_like,1,1,1,1,1569.678773,0.709641,0.709641,0.00000


In [14]:
if plot_guard() and not lifecycle.empty:
    lifecycle_status_counts = lifecycle.groupby('lifecycle_status', as_index=False).size()
    fig = px.bar(
        lifecycle_status_counts,
        x='lifecycle_status',
        y='size',
        title=f'NFPM lifecycle status counts - {POOL}',
        labels={'size': 'Positions', 'lifecycle_status': 'Lifecycle status'},
    )
    fig.show()

if plot_guard() and not lifecycle.empty:
    fig = px.histogram(
        lifecycle,
        x='observed_lifetime_days',
        color='lifecycle_status',
        nbins=60,
        title=f'Observed lifetime distribution - {POOL}',
        labels={'observed_lifetime_days': 'Observed lifetime (days)'},
    )
    fig.show()


## 4. PnL, fees et impermanent loss

In [15]:
pnl_current = query_df(dedent(
    '''
    SELECT
        token_id,
        current_owner,
        toUInt8(is_open) AS is_open,
        toUInt8(coalesce(is_active, 0)) AS is_active,
        entry_state_missing,
        current_total_wealth_token0,
        current_principal_wealth_token0,
        current_fee_wealth_token0,
        current_collected_fee_wealth_token0,
        current_uncollected_fee_wealth_token0,
        impermanent_loss_token0,
        impermanent_loss_pct,
        pnl_vs_initial_pct,
        pnl_vs_hodl_pct,
        observed_time_in_range_pct,
        fee_active_swap_count
    FROM mart_nfpm_position_pnl_current
    WHERE pool = %(pool)s
    ORDER BY current_total_wealth_token0 DESC
    LIMIT 10000
    '''
), {'pool': POOL})

if not pnl_current.empty:
    pnl_current['position_status'] = pnl_current.apply(
        lambda row: 'active' if row['is_active'] else ('open_out_of_range' if row['is_open'] else 'closed'),
        axis=1,
    )

pnl_current.head(20)


10,000 rows


,token_id,current_owner,is_open,is_active,entry_state_missing,current_total_wealth_token0,current_principal_wealth_token0,current_fee_wealth_token0,current_collected_fee_wealth_token0,current_uncollected_fee_wealth_token0,impermanent_loss_token0,impermanent_loss_pct,pnl_vs_initial_pct,pnl_vs_hodl_pct,observed_time_in_range_pct,fee_active_swap_count,position_status
0,253158,0xec890fca5c79a443f53849bc658947a99b653a4c,0,0,0,1.401378e+08,1.401378e+08,0.0,5.493473e+05,-5.493473e+05,-2.708673e+06,-1.896213e-02,4.789850e-01,-1.896213e-02,NaN,0,closed
1,923861,0x7055b17a1b911b6b971172c01ff0cc27881aea94,0,0,0,1.353184e+08,1.353184e+08,0.0,2.899384e+05,-2.899384e+05,7.205916e+04,5.327992e-04,-1.137694e-01,5.327992e-04,NaN,0,closed
2,921260,0x7055b17a1b911b6b971172c01ff0cc27881aea94,0,0,0,1.336191e+08,1.336191e+08,0.0,2.694189e+05,-2.694189e+05,-1.996756e-06,-1.494364e-14,-1.494364e-14,-1.494364e-14,NaN,0,closed
3,733923,0x7055b17a1b911b6b971172c01ff0cc27881aea94,0,0,0,1.059327e+08,1.059327e+08,0.0,2.862338e+05,-2.862338e+05,-1.996756e-06,-1.884928e-14,-1.884928e-14,-1.884928e-14,NaN,0,closed
4,190489,0xecaa8f3636270ee917c5b08d6324722c2c4951c7,0,0,0,9.572275e+07,9.572275e+07,0.0,1.610024e+06,-1.610024e+06,-6.933616e+06,-6.754200e-02,-1.818239e-01,-6.754200e-02,NaN,0,closed
5,464136,0x0000000000000000000000000000000000000000,0,0,0,9.071961e+07,9.071961e+07,0.0,9.962605e+01,-9.962605e+01,-5.515847e+04,-6.076409e-04,6.955017e-02,-6.076409e-04,NaN,0,closed
6,464233,0x0000000000000000000000000000000000000000,0,0,0,8.949967e+07,8.949967e+07,0.0,4.976316e+01,-4.976316e+01,-2.805754e+04,-3.133950e-04,6.394597e-02,-3.133950e-04,NaN,0,closed
7,405975,0x0000000000000000000000000000000000000000,0,0,0,8.920922e+07,8.920922e+07,0.0,8.596661e+01,-8.596661e+01,7.186408e+04,8.062172e-04,2.766699e-01,8.062172e-04,NaN,0,closed
8,464359,0x0000000000000000000000000000000000000000,0,0,0,8.781629e+07,8.781629e+07,0.0,3.398776e+01,-3.398776e+01,-1.879628e+04,-2.139951e-04,2.026773e-03,-2.139951e-04,NaN,0,closed
9,168412,0xecaa8f3636270ee917c5b08d6324722c2c4951c7,0,0,0,8.651152e+07,8.651152e+07,0.0,7.980570e+05,-7.980570e+05,1.268681e+07,1.718504e-01,-1.209442e-01,1.718504e-01,NaN,0,closed


In [16]:
if plot_guard() and not pnl_current.empty:
    pnl_hist = pnl_current[pnl_current['pnl_vs_hodl_pct'].notna()].copy()
    fig = px.histogram(
        pnl_hist,
        x='pnl_vs_hodl_pct',
        color='position_status',
        nbins=80,
        title=f'PnL vs HODL distribution - {POOL}',
        labels={'pnl_vs_hodl_pct': 'PnL vs HODL (%)'},
    )
    fig.show()

if plot_guard() and not pnl_current.empty:
    fig = px.scatter(
        pnl_current,
        x='current_fee_wealth_token0',
        y='pnl_vs_hodl_pct',
        color='position_status',
        hover_data=['token_id', 'current_owner', 'fee_active_swap_count'],
        title=f'Fee wealth vs PnL vs HODL - {POOL}',
        labels={'current_fee_wealth_token0': 'Fee wealth (token0)', 'pnl_vs_hodl_pct': 'PnL vs HODL (%)'},
        log_x=True,
    )
    fig.show()


In [17]:
if plot_guard() and not pnl_current.empty:
    tir = pnl_current[pnl_current['observed_time_in_range_pct'].notna() & pnl_current['pnl_vs_hodl_pct'].notna()].copy()
    fig = px.scatter(
        tir,
        x='observed_time_in_range_pct',
        y='pnl_vs_hodl_pct',
        color='position_status',
        hover_data=['token_id', 'current_owner'],
        title=f'Observed time in range vs PnL vs HODL - {POOL}',
        labels={'observed_time_in_range_pct': 'Observed time in range (%)', 'pnl_vs_hodl_pct': 'PnL vs HODL (%)'},
    )
    fig.show()

if plot_guard() and not pnl_current.empty:
    top_fee_earners = pnl_current.nlargest(20, 'current_fee_wealth_token0').copy()
    top_fee_earners['owner_short'] = top_fee_earners['current_owner'].str.slice(0, 10)
    fig = px.bar(
        top_fee_earners.sort_values('current_fee_wealth_token0'),
        x='current_fee_wealth_token0',
        y='owner_short',
        orientation='h',
        title=f'Top NFPM positions by fee wealth - {POOL}',
        labels={'current_fee_wealth_token0': 'Fee wealth (token0)', 'owner_short': 'Owner'},
        hover_data=['token_id'],
    )
    fig.show()


## 5. Quick summary

In [18]:
summary = pd.DataFrame([
    {
        'pool': POOL,
        'nfpm_rows_current': int(nfpm_snapshot.iloc[0]['positions_total']) if not nfpm_snapshot.empty and pd.notna(nfpm_snapshot.iloc[0]['positions_total']) else 0,
        'nfpm_open_positions': int(nfpm_snapshot.iloc[0]['open_positions']) if not nfpm_snapshot.empty and pd.notna(nfpm_snapshot.iloc[0]['open_positions']) else 0,
        'nfpm_active_positions': int(nfpm_snapshot.iloc[0]['active_positions']) if not nfpm_snapshot.empty and pd.notna(nfpm_snapshot.iloc[0]['active_positions']) else 0,
        'nfpm_owner_count': int(nfpm_snapshot.iloc[0]['current_owner_count']) if not nfpm_snapshot.empty and pd.notna(nfpm_snapshot.iloc[0]['current_owner_count']) else 0,
        'pool_total_tvl_token0': float(pool_context.iloc[0]['total_tvl_token0']) if not pool_context.empty else None,
        'nfpm_deployed_principal_token0': float(nfpm_snapshot.iloc[0]['deployed_principal_token0']) if not nfpm_snapshot.empty and pd.notna(nfpm_snapshot.iloc[0]['deployed_principal_token0']) else None,
        'nfpm_total_wealth_token0': float(nfpm_pnl_snapshot.iloc[0]['total_current_wealth_token0']) if not nfpm_pnl_snapshot.empty and pd.notna(nfpm_pnl_snapshot.iloc[0]['total_current_wealth_token0']) else None,
        'nfpm_total_fee_wealth_token0': float(nfpm_pnl_snapshot.iloc[0]['total_fee_wealth_token0']) if not nfpm_pnl_snapshot.empty and pd.notna(nfpm_pnl_snapshot.iloc[0]['total_fee_wealth_token0']) else None,
        'nfpm_total_uncollected_fee_wealth_token0': float(nfpm_pnl_snapshot.iloc[0]['total_uncollected_fee_wealth_token0']) if not nfpm_pnl_snapshot.empty and pd.notna(nfpm_pnl_snapshot.iloc[0]['total_uncollected_fee_wealth_token0']) else None,
        'avg_observed_time_in_range_pct': float(nfpm_pnl_snapshot.iloc[0]['avg_observed_time_in_range_pct']) if not nfpm_pnl_snapshot.empty and pd.notna(nfpm_pnl_snapshot.iloc[0]['avg_observed_time_in_range_pct']) else None,
        'median_pnl_vs_hodl_pct': float(nfpm_pnl_snapshot.iloc[0]['median_pnl_vs_hodl_pct']) if not nfpm_pnl_snapshot.empty and pd.notna(nfpm_pnl_snapshot.iloc[0]['median_pnl_vs_hodl_pct']) else None,
    }
])

if not summary.empty and pd.notna(summary.loc[0, 'pool_total_tvl_token0']) and pd.notna(summary.loc[0, 'nfpm_deployed_principal_token0']) and summary.loc[0, 'pool_total_tvl_token0']:
    summary['nfpm_deployed_vs_pool_tvl_pct'] = summary['nfpm_deployed_principal_token0'] / summary['pool_total_tvl_token0']

summary


,pool,nfpm_rows_current,nfpm_open_positions,nfpm_active_positions,nfpm_owner_count,pool_total_tvl_token0,nfpm_deployed_principal_token0,nfpm_total_wealth_token0,nfpm_total_fee_wealth_token0,nfpm_total_uncollected_fee_wealth_token0,avg_observed_time_in_range_pct,median_pnl_vs_hodl_pct,nfpm_deployed_vs_pool_tvl_pct
0,0x88e6A0c2dDD26FEEb64F039a2c41296FcB3f5640,114358,4058,2205,22609,9.362351e+07,9.345948e+07,7.193793e+11,0.0,-2.738792e+08,None,-9.999785e-12,0.998248
